# Lesson 03 - Agentsete disainimustrite kujundamine

Selles õppetükis uurime kolme põhjalikku disainimustrit tõhusate AI-agentide loomiseks:

1. **Selged agentide juhised** — täpsete ja rolli määravate juhiste koostamine agentide käitumise suunamiseks
2. **Struktureeritud väljund Pydantic mudelitega** — tagades agentide poolt ennustatava ja valideeritud andmete tagastamise
3. **Ühe vastutusega agentide disainimine** — keskendunud agentide loomine, mis teevad igaüks üht asja hästi

Rakendame iga mustrit **reisisihtkoha soovitaja** stsenaariumile, ehitades järk-järgult süsteemi, mis suudab soovitada sihtkohti, kontrollida saadavust ja hallata logistikat.


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Muster 1: Selged Agendi Juhised

Kõige mõjusam muster on ka kõige lihtsam: kirjutada agendile selged, detailirohked juhised.

Head juhised määratlevad:
- **Kes** agent on (isiksus ja toon)
- **Mida** ta peaks tegema (samm-sammult kohustused)
- **Kuidas** ta peaks käituma (piirangud ja stiil)

Allpool loome reisikonsjerži agendi, kellel on selged juhised, mis kujundavad iga tema vastuse.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Muster 2: Struktureeritud väljund Pydantic mudelitega

Vaba kujuga tekst on kasulik vestluseks, kuid alluvad süsteemid vajavad struktureeritud andmeid.  
Paaritades **Pydantic mudeleid** ja **tööriista funktsiooni**, saame:

- Määratleda agendi väljundi täpse skeemi  
- Vastuseid automaatselt valideerida  
- Agendi tulemusi rakenduse loogikasse usaldusväärselt integreerida  

Samuti tutvustame tööriista, mis tagastab sihtkoha üksikasjad, et agent saaks oma soovitused pärisandmetel põhineda.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Muster 3: Ühe vastutusega agendid

Komplekssed ülesanded saavad kasu töö jaotamisest mitme fookustatud agendi vahel, kellel kõigil on üksainus vastutusala:

- **Sihtkoha ekspert**, kes teab kohti ja saadavust
- **Logistika planeerija**, kes korraldab lende, hotelle ja marsruute

See peegeldab tarkvaraarenduse põhimõtet *huvide lahusolek* — iga agenti on lihtsam iseseisvalt testida, hooldada ja täiustada.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Kokkuvõte

Selles tunnis rakendasime kolme agendi disainimustrit reisisõltlase soovitussüsteemi stsenaariumil:

| Muster | Põhitähtsus | Kasu |
|---|---|---|
| **Selged juhised** | Määratle persona, ülesanded ja piirangud ette | Järjekindel, brändile vastav agendi käitumine |
| **Struktureeritud väljund** | Kasuta vastuse vorminguks Pydanticu mudeleid | Kinnitatud, masinloetavad tulemused |
| **Üks vastutus** | Too igale agendile üks konkreetne ülesanne | Lihtsam testida, hooldada ja kombineerida |

Need mustrid sobivad loomulikult kokku — saate kombineerida selgeid juhiseid struktureeritud väljundiga ühe vastutusega agendi sees, et luua kindlaid ja tootmiskõlblikke süsteeme.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
